In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
# Класс единичного формального нейрона
class FormalNeuron:
    def __init__(self, n_inputs, activation_function):
        self.weights = np.random.randn(n_inputs)
        self.bias = np.random.randn()
        self.activation_function = activation_function

    def forward(self, x):
        linear_output = np.dot(x, self.weights) + self.bias
        return self.activation_function(linear_output)

    def update_weights(self, new_weights, new_bias):
        self.weights = new_weights
        self.bias = new_bias

In [ ]:
# Функции активации
sigmoid = lambda x: 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

In [ ]:
# Загружаем датасет грибов (UCI Mushroom Dataset)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"
columns = [
    'class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
    'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
    'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
    'stalk-surface-below-ring', 'stalk-color-above-ring',
    'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
    'ring-type', 'spore-print-color', 'population', 'habitat'
]
data = pd.read_csv(url, header=None, names=columns)
data.head()

In [ ]:
# Обработка данных — кодируем все категориальные признаки
le = LabelEncoder()
for col in data.columns:
    data[col] = le.fit_transform(data[col])

# Целевая переменная: 1 = ядовитый, 0 = съедобный
X = data.drop('class', axis=1).values
y = data['class'].values.reshape(-1, 1)

print(f"Датасет: {X.shape[0]} грибов, {X.shape[1]} признаков")
print(f"Съедобных: {(y == 0).sum()}, ядовитых: {(y == 1).sum()}")

In [ ]:
# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Создаём нейрон
np.random.seed(42)
neuron = FormalNeuron(n_inputs=X_train.shape[1], activation_function=sigmoid)

# Гиперпараметры
epochs = 15
learning_rate = 0.1
loss_history = []

# Обучение через эпохи
for epoch in range(epochs):
    total_loss = 0
    for i in range(len(X_train)):
        x_sample = X_train[i]
        y_sample = y_train[i]

        # Прямой проход
        output = neuron.forward(x_sample)
        loss = (y_sample - output) ** 2
        total_loss += loss

        # Обратное распространение
        error = y_sample - output
        delta = error * sigmoid_derivative(output)

        neuron.weights += learning_rate * delta * x_sample
        neuron.bias += learning_rate * delta

    loss_history.append(total_loss[0])
    print(f"Epoch {epoch}, Loss: {total_loss[0]:.4f}")

In [ ]:
# График ошибки
plt.plot(loss_history)
plt.xlabel("Epochs")
plt.ylabel("Total Loss")
plt.title("Training Loss over Epochs")
plt.show()

In [ ]:
# Оценка на тесте
predictions_test = np.array([neuron.forward(x) > 0.5 for x in X_test])
test_accuracy = np.mean(predictions_test.flatten() == y_test.flatten())

print(f'Test Accuracy: {test_accuracy:.4f}')